# Independent external speech evaluation
PSRB public sample and Persian YouTube speech remain held out from training, development, and model selection. Run source preparation explicitly, then evaluate the same three-seed matched models and the matched-update 64-hour control. No training starts here.

See README.md for audio acquisition and YouTube episode-manifest preparation.

In [ ]:
# Resolve shared code when launched from code/ or the repository root.
import sys
from pathlib import Path

_code_candidates = [Path.cwd(), Path.cwd() / "code"]
CODE = next(
    (
        p.resolve()
        for p in _code_candidates
        if (p / "neyshekar_experiments" / "protocol.py").is_file()
    ),
    None,
)
if CODE is None:
    raise RuntimeError("Launch this notebook from the repository root or its code/ directory.")
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
from neyshekar_experiments.protocol import ROOT
# End notebook bootstrap

from neyshekar_experiments.external import prepare_psrb, external_sets
from neyshekar_experiments.training import (
    experiment_grid,
    evaluate_run,
    evaluate_zero_shot,
    ZERO_SHOT_MODELS,
)
from neyshekar_experiments.reporting import load_scores

RUN_PREPARATION = False
RUN_EVALUATION = False
DATASETS = ["psrb_sample"]  # Add "youtube_timestamps" after preparing its manifest.

In [ ]:
if RUN_PREPARATION:
    summary = prepare_psrb(download=True)
    print({key: summary[key] for key in ("clips", "hours", "sha256")})

## Frozen test sets
Audio/reference hashes and segmentation boundaries are verified. All subset statistics and paired bootstrap comparisons use saved full-test predictions. PSRB uses released clips; YouTube uses reference-timestamp boundaries and episode-clustered resampling. Long audio is retained, so long CTC segments may require substantial GPU memory.

In [ ]:
if RUN_EVALUATION:
    external_sets(DATASETS)
    runs = experiment_grid("matched") + experiment_grid("mixture_updates")
    missing = [
        run.name
        for run in runs
        if not (ROOT / "checkpoints/v2" / run.name / "complete.json").exists()
    ]
    if missing:
        raise RuntimeError(f"Complete training first: {missing}")
    for model in ZERO_SHOT_MODELS:
        evaluate_zero_shot(model, external=DATASETS, internal=False)
    for run in runs:
        evaluate_run(run, ROOT / "checkpoints/v2" / run.name, external=DATASETS, internal=False)

In [ ]:
scores = load_scores()
scores[scores.evaluation.str.startswith(("psrb_", "youtube_"))]

Run `python -m neyshekar_experiments analyze` and `python -m neyshekar_experiments tables` after the grids complete. Tables require three seeds for controlled comparisons. No external score is available until decoding runs.